# 02b_expert_agreement_analysis

This notebook quantifies inter-expert agreement for expert hotspot annotations.

It is intentionally standalone and keeps the staged notebook structure.


Clinical labels come from the folder structure. Expert XML annotations are used here only for hotspot presence and localization agreement.

## 1. Imports and paths

In [1]:
from pathlib import Path
import os, json, shutil, zipfile, hashlib, re, warnings
from datetime import datetime, timezone
import pandas as pd
import numpy as np


BASE_DIR = Path("/content")
PROJECT_NAME = "project_thermography_equine"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME

DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
MODEL_SELECTION_DIR = OUTPUT_ROOT / "model_selection"

for p in [PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR, ANNOTATIONS_DIR,
          OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, MODEL_SELECTION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project paths initialized")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPLIT_DATA_DIR:", SPLIT_DATA_DIR)
print("ANNOTATIONS_DIR:", ANNOTATIONS_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

import xml.etree.ElementTree as ET
from sklearn.metrics import cohen_kappa_score

Project paths initialized
PROJECT_ROOT: /content/project_thermography_equine
SPLIT_DATA_DIR: /content/project_thermography_equine/data/dataset_split
ANNOTATIONS_DIR: /content/project_thermography_equine/data/annotations
OUTPUT_ROOT: /content/project_thermography_equine/outputs


## 2. Load required upstream outputs

In [2]:
required = [
    CONFIG_DIR / "analysis_config.json",
    CONFIG_DIR / "study_protocol.json",
    CONFIG_DIR / "master_metadata.csv",
    CONFIG_DIR / "expert_image_hotspot_status.csv",
    CONFIG_DIR / "hotspot_bboxes.csv",
    CONFIG_DIR / "hotspot_points.csv",
    CONFIG_DIR / "hotspot_consensus_by_image.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Run notebooks 00, 01 and 02 first. Missing:\n" + "\n".join(missing))

# 02 should create this file explicitly. For backward compatibility with older 02 versions,
# rebuild it here from master_metadata.csv + hotspot_consensus_by_image.csv if needed.
master_ann_path = CONFIG_DIR / "master_metadata_with_annotations.csv"
if not master_ann_path.exists():
    master_base = pd.read_csv(CONFIG_DIR / "master_metadata.csv")
    consensus_for_master = pd.read_csv(CONFIG_DIR / "hotspot_consensus_by_image.csv")
    ann_cols = [
        "split", "image_name",
        "has_hotspot_expert1", "has_hotspot_expert2", "has_hotspot_any_expert",
        "has_hotspot_both_experts", "expert_hotspot_disagreement",
        "healthy_with_expert_hotspot", "annotation_label_conflict", "annotation_clinical_note"
    ]
    available_ann_cols = [c for c in ann_cols if c in consensus_for_master.columns]
    rebuilt = master_base.merge(
        consensus_for_master[available_ann_cols].drop_duplicates(subset=["split", "image_name"]),
        on=["split", "image_name"],
        how="left"
    )
    for col in [c for c in available_ann_cols if c not in ["split", "image_name", "annotation_clinical_note"]]:
        rebuilt[col] = rebuilt[col].fillna(False).astype(bool)
    if "annotation_clinical_note" in rebuilt.columns:
        rebuilt["annotation_clinical_note"] = rebuilt["annotation_clinical_note"].fillna("no_clinical_annotation_conflict")
    rebuilt.to_csv(master_ann_path, index=False)
    rebuilt.to_csv(REPORTS_DIR / "master_metadata_with_annotations.csv", index=False)
    print("Rebuilt missing master_metadata_with_annotations.csv from master_metadata.csv and hotspot_consensus_by_image.csv")

with open(CONFIG_DIR / "analysis_config.json", "r", encoding="utf-8") as f:
    analysis_config = json.load(f)
with open(CONFIG_DIR / "study_protocol.json", "r", encoding="utf-8") as f:
    study_protocol = json.load(f)

master = pd.read_csv(master_ann_path)
status = pd.read_csv(CONFIG_DIR / "expert_image_hotspot_status.csv")
bboxes = pd.read_csv(CONFIG_DIR / "hotspot_bboxes.csv")
points = pd.read_csv(CONFIG_DIR / "hotspot_points.csv")

print("Master rows:", len(master))
print("Expert image-status rows:", len(status))
print("BBoxes:", len(bboxes), "Points:", len(points))
display(master.groupby(["split", "label_clinical"]).size().reset_index(name="n"))


Master rows: 347
Expert image-status rows: 694
BBoxes: 192 Points: 192


,split,label_clinical,n
0,test,healthy,40
1,test,pathological,13
2,train,healthy,179
3,train,pathological,63
4,valid,healthy,38
5,valid,pathological,14


## 3. Helper functions

In [3]:
def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = float(a["xtl"]), float(a["ytl"]), float(a["xbr"]), float(a["ybr"])
    bx1, by1, bx2, by2 = float(b["xtl"]), float(b["ytl"]), float(b["xbr"]), float(b["ybr"])
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = area_a + area_b - inter
    return np.nan if denom <= 0 else inter / denom

def bbox_center(row):
    return ((float(row["xtl"]) + float(row["xbr"])) / 2.0, (float(row["ytl"]) + float(row["ybr"])) / 2.0)

def euclidean(p1, p2):
    return float(np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2))

def largest_box_per_image(df):
    if df.empty:
        return df.copy()
    d = df.copy()
    if "bbox_area" not in d.columns:
        d["bbox_area"] = (d["xbr"] - d["xtl"]).clip(lower=0) * (d["ybr"] - d["ytl"]).clip(lower=0)
    d = d.sort_values(["split", "image_name", "expert", "bbox_area"], ascending=[True, True, True, False])
    return d.groupby(["split", "image_name", "expert"], as_index=False).head(1).reset_index(drop=True)

def first_point_per_image(df):
    if df.empty:
        return df.copy()
    d = df.copy()
    return d.sort_values(["split", "image_name", "expert"]).groupby(["split", "image_name", "expert"], as_index=False).head(1).reset_index(drop=True)

## 4. Presence agreement

In [4]:

presence = status.copy()
presence["expert_has_hotspot"] = presence["expert_has_hotspot"].astype(bool)

presence_wide = presence.pivot_table(
    index=["split", "image_name", "horse_id", "relative_image_path", "folder_label", "label_clinical", "label_binary"],
    columns="expert",
    values="expert_has_hotspot",
    aggfunc="max",
    fill_value=False,
).reset_index()
for e in ["expert1", "expert2"]:
    if e not in presence_wide.columns:
        presence_wide[e] = False
presence_wide["expert1"] = presence_wide["expert1"].astype(bool)
presence_wide["expert2"] = presence_wide["expert2"].astype(bool)
presence_wide["presence_agreement"] = presence_wide["expert1"] == presence_wide["expert2"]
presence_wide["both_hotspot"] = presence_wide["expert1"] & presence_wide["expert2"]
presence_wide["either_hotspot"] = presence_wide["expert1"] | presence_wide["expert2"]

presence_rows = []
for split_name, d in list(presence_wide.groupby("split")) + [("overall", presence_wide)]:
    if len(d) == 0:
        continue
    try:
        kappa = cohen_kappa_score(d["expert1"].astype(int), d["expert2"].astype(int))
    except Exception:
        kappa = np.nan
    presence_rows.append({
        "split": split_name,
        "n_images": len(d),
        "expert1_hotspot_n": int(d["expert1"].sum()),
        "expert2_hotspot_n": int(d["expert2"].sum()),
        "both_hotspot_n": int(d["both_hotspot"].sum()),
        "either_hotspot_n": int(d["either_hotspot"].sum()),
        "agreement_n": int(d["presence_agreement"].sum()),
        "agreement_rate": float(d["presence_agreement"].mean()),
        "cohen_kappa": float(kappa) if pd.notna(kappa) else np.nan,
    })

presence_summary = pd.DataFrame(presence_rows)
display(presence_summary)

,split,n_images,expert1_hotspot_n,expert2_hotspot_n,both_hotspot_n,either_hotspot_n,agreement_n,agreement_rate,cohen_kappa
0,test,53,19,19,19,19,53,1.0,1.0
1,train,242,63,63,63,63,242,1.0,1.0
2,valid,52,14,14,14,14,52,1.0,1.0
3,overall,347,96,96,96,96,347,1.0,1.0


## 5. Localization agreement: bounding boxes and points

In [5]:
# Use one representative object per expert-image pair: largest box and first point.
box1 = largest_box_per_image(bboxes[bboxes["expert"] == "expert1"])
box2 = largest_box_per_image(bboxes[bboxes["expert"] == "expert2"])
pt1 = first_point_per_image(points[points["expert"] == "expert1"])
pt2 = first_point_per_image(points[points["expert"] == "expert2"])

paired_boxes = box1.merge(
    box2,
    on=["split", "image_name"],
    suffixes=("_expert1", "_expert2"),
    how="inner",
)

loc_rows = []
for _, r in paired_boxes.iterrows():
    a = {"xtl": r["xtl_expert1"], "ytl": r["ytl_expert1"], "xbr": r["xbr_expert1"], "ybr": r["ybr_expert1"]}
    b = {"xtl": r["xtl_expert2"], "ytl": r["ytl_expert2"], "xbr": r["xbr_expert2"], "ybr": r["ybr_expert2"]}
    ca, cb = bbox_center(a), bbox_center(b)
    loc_rows.append({
        "split": r["split"],
        "image_name": r["image_name"],
        "bbox_iou_expert1_expert2": bbox_iou(a, b),
        "bbox_center_distance_px": euclidean(ca, cb),
        "bbox_area_expert1": float(r.get("bbox_area_expert1", np.nan)),
        "bbox_area_expert2": float(r.get("bbox_area_expert2", np.nan)),
        "image_width": float(r.get("image_width_expert1", np.nan)),
        "image_height": float(r.get("image_height_expert1", np.nan)),
    })

localization_by_image = pd.DataFrame(loc_rows)

paired_points = pt1.merge(
    pt2,
    on=["split", "image_name"],
    suffixes=("_expert1", "_expert2"),
    how="inner",
)
if not paired_points.empty:
    paired_points["point_distance_px"] = np.sqrt(
        (paired_points["x_expert1"] - paired_points["x_expert2"])**2 +
        (paired_points["y_expert1"] - paired_points["y_expert2"])**2
    )
    localization_by_image = localization_by_image.merge(
        paired_points[["split", "image_name", "point_distance_px"]],
        on=["split", "image_name"], how="left"
    )
else:
    localization_by_image["point_distance_px"] = np.nan

# Add clinical/folder metadata.
localization_by_image = localization_by_image.merge(
    master[["split", "image_name", "horse_id", "relative_image_path", "label_clinical", "label_binary"]],
    on=["split", "image_name"], how="left"
)

display(localization_by_image.head())

,split,image_name,bbox_iou_expert1_expert2,bbox_center_distance_px,bbox_area_expert1,bbox_area_expert2,image_width,image_height,point_distance_px,horse_id,relative_image_path,label_clinical,label_binary
0,test,0A8XC.jpg,0.822615,3.596585,720.6300,688.1310,492.0,331.0,6.438167,0A8XC,test/injured/0A8XC.jpg,pathological,1
1,test,0JNGF.jpg,0.517047,5.523713,329.7000,210.8240,492.0,331.0,5.177200,0JNGF,test/injured/0JNGF.jpg,pathological,1
2,test,6Y0WA.jpg,0.899866,2.040000,1164.9226,1294.5510,492.0,331.0,29.329393,6Y0WA,test/injured/6Y0WA.jpg,pathological,1
3,test,73Q1G.jpg,0.718227,1.842532,216.3136,277.9852,492.0,331.0,0.000000,73Q1G,test/healthy/73Q1G.jpg,healthy,0
4,test,7KL5H.jpg,0.865715,1.598789,1145.9136,1145.6640,492.0,331.0,3.600222,7KL5H,test/injured/7KL5H.jpg,pathological,1


## 6. Agreement summaries and flags

In [6]:
loc_rows = []
if localization_by_image.empty:
    localization_summary = pd.DataFrame(columns=["split", "n_paired_boxes", "mean_iou", "median_iou", "mean_point_distance_px", "median_point_distance_px"])
else:
    for split_name, d in list(localization_by_image.groupby("split")) + [("overall", localization_by_image)]:
        loc_rows.append({
            "split": split_name,
            "n_paired_boxes": int(d["bbox_iou_expert1_expert2"].notna().sum()),
            "mean_iou": float(d["bbox_iou_expert1_expert2"].mean()),
            "median_iou": float(d["bbox_iou_expert1_expert2"].median()),
            "q1_iou": float(d["bbox_iou_expert1_expert2"].quantile(0.25)),
            "q3_iou": float(d["bbox_iou_expert1_expert2"].quantile(0.75)),
            "low_iou_lt_0_25_n": int((d["bbox_iou_expert1_expert2"] < 0.25).sum()),
            "mean_point_distance_px": float(d["point_distance_px"].mean()) if "point_distance_px" in d else np.nan,
            "median_point_distance_px": float(d["point_distance_px"].median()) if "point_distance_px" in d else np.nan,
        })
    localization_summary = pd.DataFrame(loc_rows)

display(localization_summary)

low_iou_cases = localization_by_image[localization_by_image["bbox_iou_expert1_expert2"] < 0.25].copy()
display(low_iou_cases[["split", "image_name", "bbox_iou_expert1_expert2", "relative_image_path"]].head(30))

,split,n_paired_boxes,mean_iou,median_iou,q1_iou,q3_iou,low_iou_lt_0_25_n,mean_point_distance_px,median_point_distance_px
0,test,19,0.830807,0.885517,0.723039,1.000000,0,3.573597,0.0
1,train,63,0.785939,0.802964,0.680295,0.913593,0,4.892115,0.0
2,valid,14,0.730945,0.710870,0.688903,0.791344,0,4.491144,0.0
3,overall,96,0.786799,0.798276,0.686067,0.903621,0,4.572683,0.0


,split,image_name,bbox_iou_expert1_expert2,relative_image_path


## 7. Save outputs

In [7]:
presence_wide.to_csv(CONFIG_DIR / "expert_agreement_presence_by_image.csv", index=False)
presence_wide.to_csv(REPORTS_DIR / "expert_agreement_presence_by_image.csv", index=False)
presence_summary.to_csv(CONFIG_DIR / "expert_agreement_presence_summary.csv", index=False)
presence_summary.to_csv(REPORTS_DIR / "expert_agreement_presence_summary.csv", index=False)
localization_by_image.to_csv(CONFIG_DIR / "expert_agreement_localization_by_image.csv", index=False)
localization_by_image.to_csv(REPORTS_DIR / "expert_agreement_localization_by_image.csv", index=False)
localization_summary.to_csv(CONFIG_DIR / "expert_agreement_localization_summary.csv", index=False)
localization_summary.to_csv(REPORTS_DIR / "expert_agreement_localization_summary.csv", index=False)
low_iou_cases.to_csv(REPORTS_DIR / "expert_agreement_low_iou_cases.csv", index=False)


summary_parts = []
for _, row in presence_summary.iterrows():
    summary_parts.append({"section": "presence", **row.to_dict()})
for _, row in localization_summary.iterrows():
    summary_parts.append({"section": "localization", **row.to_dict()})
expert_agreement_summary = pd.DataFrame(summary_parts)
expert_agreement_summary.to_csv(TABLES_DIR / "table_expert_agreement_summary.csv", index=False)
expert_agreement_summary.to_csv(REPORTS_DIR / "expert_agreement_summary.csv", index=False)

print("Saved expert agreement outputs.")
print("Low-IoU cases:", len(low_iou_cases))

Saved expert agreement outputs.
Low-IoU cases: 0


## 8. Methods/results text seed

In [8]:
overall_presence = presence_summary[presence_summary["split"] == "overall"].iloc[0]
if not localization_summary.empty and "overall" in set(localization_summary["split"]):
    overall_loc = localization_summary[localization_summary["split"] == "overall"].iloc[0]
    loc_sentence = f"For images annotated by both experts, median bounding-box IoU was {overall_loc['median_iou']:.3f}."
else:
    loc_sentence = "Localization agreement could not be summarized because no paired expert boxes were available."

expert_agreement_text = (
    f"Inter-expert hotspot-presence agreement was {overall_presence['agreement_rate']:.3f} "
    f"(Cohen's kappa={overall_presence['cohen_kappa']:.3f}) across {int(overall_presence['n_images'])} images. "
    + loc_sentence
)

(CONFIG_DIR / "methods_expert_agreement_text.txt").write_text(expert_agreement_text, encoding="utf-8")
(REPORTS_DIR / "methods_expert_agreement_text.txt").write_text(expert_agreement_text, encoding="utf-8")
print(expert_agreement_text)

Inter-expert hotspot-presence agreement was 1.000 (Cohen's kappa=1.000) across 347 images. For images annotated by both experts, median bounding-box IoU was 0.798.
